# Module 1: Meet Jev: Typed Decisions Instead of Text

Your first Jev calls, side by side with an LLM.

## Learning Objectives

1. **Call** Jev from Python with Choice, Score and Noul questions in one request
2. **Read** probabilities and confidence, and **pin** the model version
3. **Compare** Jev with an LLM on the same triage question: output, speed and cost
4. **Turn** Jev judgments into an action with a truth table in code

## Prerequisites

- Python in notebooks and basic classification metrics
- A Jev API key and an OpenAI API key (given in class)

## Session Format

| Section | Type |
|---|---|
| Setup | Run together |
| Demo 1: The same question to an LLM and to Jev | Instructor demo, you run along |
| Demo 2: One request, three question types | Instructor demo, you run along |
| Lab 1: Triage in one call | You build it, three levels |

## The Story So Far

This is where the story starts. Our running case is CLINC150: 23,850 messages people typed
to a personal assistant, across 10 domains and 150 intents, plus out-of-scope messages. Over
four modules we build the triage layer that reads each message and decides where it goes.

# Section 0: Setup

Run the next four cells in order. The key cell asks for each key once. Keys stay in this
runtime only and are never printed.

In [ ]:
%pip install -q typesafe-sdk==0.7.5 openai==3.41.0 datasets==5.1.0

In [ ]:
# ===========================================================================
# KEYS: Jev and OpenAI, typed once, never printed
# ===========================================================================
# Why: keys never live in the notebook. If your environment already has them, we use them.
import os
from getpass import getpass

for key_name in ("TYPESAFE_API_KEY", "OPENAI_API_KEY"):
    if not os.environ.get(key_name):
        os.environ[key_name] = getpass(f"{key_name}: ")
    if not os.environ[key_name].strip():
        raise ValueError(f"{key_name} is empty. Ask your instructor for the class key.")
print("Both keys are set for this runtime.")

In [ ]:
# ===========================================================================
# CONFIG: clients, pinned models, prices, one probe call
# ===========================================================================
# Why: pinned model ids keep today's numbers reproducible next month.
# The whole room shares one key. typesafe-sdk retries a 408, 429 or 5xx twice with backoff
# (its default RetryPolicy) and the OpenAI SDK retries a 429 too. A call that still fails
# raises an error: wait a moment and run the cell again.
import statistics
import time

from openai import OpenAI
from typesafe_sdk import Choice, Noul, Score, TypeSafeClient

JEV_MODEL = "jev-1.14.0"                 # pinned; "jev-latest" moves with each release
LLM_MODEL = "gpt-5.4-mini-2026-03-17"    # pinned OpenAI snapshot
JEV_PRICE_IN = 0.042 / 1_000_000         # USD per input token; Jev output is free
LLM_PRICE_IN = 0.75 / 1_000_000          # USD per input token
LLM_PRICE_OUT = 4.50 / 1_000_000         # USD per output token

jev = TypeSafeClient()
llm = OpenAI()

probe = jev.system_one(state={"message": "hello there"}, model=JEV_MODEL,
                       questions={"greeting": Noul(instructions="Is `message` a greeting?")})
assert probe.model == JEV_MODEL, f"expected {JEV_MODEL}, got {probe.model}"
llm.models.retrieve(LLM_MODEL)
print(f"Jev answered with {probe.model}: greeting = {probe.answers['greeting'].noul:.2f}")
print(f"OpenAI model {LLM_MODEL} is available.")

In [ ]:
# ===========================================================================
# DATA: CLINC150 messages and the 10 teams
# ===========================================================================
# Why: CLINC150 labels intents. The authors' domains.json maps each intent to one of 10
# domains; we treat each domain as a team in our support org. Both sources are pinned: the
# oos-eval commit in the URL and the Hugging Face dataset revision.
import json
import random
import urllib.request

from datasets import load_dataset

DOMAINS_URL = ("https://raw.githubusercontent.com/clinc/oos-eval/"
               "828f8093932c8fe6ca7936c3d2e52903b1c523de/data/domains.json")
try:
    domains = json.load(urllib.request.urlopen(DOMAINS_URL, timeout=30))
except OSError as error:
    raise RuntimeError(f"Could not download the CLINC150 domain map ({error}). "
                       "Check that the runtime is online and run this cell again.") from error
intent_to_team = {intent: team for team, intents in domains.items() for intent in intents}

clinc = load_dataset("clinc/clinc_oos", "plus", split="test",
                     revision="155b9c710419136e17307b80d0a13e68cd46b4ec")
intent_names = clinc.features["intent"].names
in_scope = [(row["text"], intent_to_team[intent_names[row["intent"]]])
            for row in clinc if intent_names[row["intent"]] != "oos"]

TEAMS = {
    "banking": "accounts, balances, transfers, bills, direct deposit",
    "credit_cards": "credit cards: limits, rewards, lost or damaged cards, APR",
    "kitchen_and_dining": "recipes, cooking, nutrition, restaurants",
    "home": "smart home, shopping and todo lists, reminders, calendar",
    "auto_and_commute": "car maintenance, gas, traffic, rides, directions",
    "travel": "flights, hotels, luggage, visas, travel alerts, translation",
    "utility": "time, date, weather, timers, calculator, definitions",
    "work": "payroll, PTO, taxes, 401k, meetings",
    "small_talk": "greetings, jokes, chit-chat with the assistant",
    "meta": "settings of the assistant itself: name, language, voice, volume",
}

rng = random.Random(42)
sample = []
for team in TEAMS:
    sample += rng.sample([pair for pair in in_scope if pair[1] == team], 4)

print(f"{len(clinc)} test messages, {len(in_scope)} in scope; our sample: {len(sample)} (4 per team)")
for text, team in sample[:3]:
    print(f"  {team:14} | {text}")

## Demo 1: The same question to an LLM and to Jev

Came here from the slides? Run the four Setup cells above first (install, keys, config, data).

An LLM writes text; Jev answers a typed question. We give both the same 40 messages, the
same 10 teams and the same team descriptions. The LLM goes twice: first a plain prompt with
no format line (on purpose, and the cell says so), then the fix every OpenAI user knows, a
JSON schema. Watch what comes back, how long it takes, and what it costs.

<!-- DIAGRAM: llm-vs-jev -->
![The same 40 CLINC150 messages go to gpt-5.4-mini, first with no format line and then with a JSON schema that always returns a team but no probabilities, and to Jev, which returns a team plus a probability per team; all three compared on accuracy, latency and cost](https://raw.githubusercontent.com/axelsirota/getting_started_jev/main/course/diagrams/module_01/llm-vs-jev.png)

In [ ]:
# ===========================================================================
# DEMO: ask the LLM which team should handle a message (NARRATE)
# ===========================================================================
# Why: this is how many teams start: a prompt that lists the options. Read what comes back.
# I am forcing your hand, and I say so: this prompt has no format line. With "Reply with
# only the team name" added, the same model gave 40 of 40 exact names in our run on
# 2026-10-30. The schema in the next cell turns that hope into a guarantee, and the schema
# row is the fair comparison with Jev.
TEAM_LIST = "\n".join(f"- {name}: {desc}" for name, desc in TEAMS.items())
TEAM_SCHEMA = {"type": "json_schema", "json_schema": {"name": "triage", "strict": True, "schema": {
    "type": "object", "properties": {"team": {"type": "string", "enum": list(TEAMS)}},
    "required": ["team"], "additionalProperties": False}}}


def ask_llm(message: str, schema: bool = False) -> dict:
    """Ask gpt-5.4-mini for the team. schema=True forces a JSON reply with one of the teams."""
    prompt = ("You triage messages for a personal assistant app. Which team should handle "
              f"this message?\nTeams:\n{TEAM_LIST}\n\nMessage: {message}")
    extra = {}
    if schema:
        prompt += "\n\nReply with only the team name, exactly as written above."
        extra = {"response_format": TEAM_SCHEMA}
    start = time.perf_counter()
    reply = llm.chat.completions.create(model=LLM_MODEL, reasoning_effort="none",
                                        messages=[{"role": "user", "content": prompt}], **extra)
    ms = (time.perf_counter() - start) * 1000
    text = reply.choices[0].message.content.strip()
    team = json.loads(text)["team"] if schema else (text if text in TEAMS else None)
    usage = reply.usage
    return {"text": text, "team": team, "ms": ms,
            "cost": usage.prompt_tokens * LLM_PRICE_IN + usage.completion_tokens * LLM_PRICE_OUT}


query = "i lost my credit card and i fly to rome tomorrow"
answer = ask_llm(query)
print(f"=== LLM, no format line ({answer['ms']:.0f} ms) ===")
print(repr(answer["text"]))
print("exact team name?", answer["team"] is not None)

In [ ]:
# ===========================================================================
# DEMO: the fix every OpenAI user knows: a JSON schema (NARRATE)
# ===========================================================================
# Why: an enum in the schema forces one of the 10 team names. The format problem is gone.
# What is still missing: any sign of how sure the model was.
answer = ask_llm(query, schema=True)
print(f"=== LLM, JSON schema ({answer['ms']:.0f} ms) ===")
print(repr(answer["text"]), "->", answer["team"])

In [ ]:
# ===========================================================================
# DEMO: ask Jev the same question (NARRATE)
# ===========================================================================
# Why: the options are data, the answer is typed, and it comes with a probability per team.
team_question = Choice(instructions="Which team should handle `message`?", criteria=TEAMS)


def ask_jev(message: str) -> dict:
    start = time.perf_counter()
    response = jev.system_one(state={"message": message}, model=JEV_MODEL,
                              questions={"team": team_question})
    ms = (time.perf_counter() - start) * 1000
    team = response.answers["team"]
    return {"team": team.choice, "confidence": team.confidence,
            "probabilities": team.probabilities, "ms": ms,
            "cost": response.usage.input_tokens * JEV_PRICE_IN}


answer = ask_jev(query)
print(f"=== JEV ({answer['ms']:.0f} ms) ===")
print(f"team: {answer['team']}   confidence: {answer['confidence']:.2f}")
top3 = sorted(answer["probabilities"].items(), key=lambda kv: -kv[1])[:3]
print("top 3:", ", ".join(f"{name} {p:.2f}" for name, p in top3))

In [ ]:
# ===========================================================================
# DEMO: 40 messages, three ways, one table (NARRATE)
# ===========================================================================
# Why: one example proves nothing. Same 40 messages, measured side by side. Accuracy only
# counts an exact team name; anything else is wrong, as it would be for your code.
# 120 calls on a shared key: progress every 10 messages, and a call that still fails after
# the SDK retries is counted and skipped instead of stopping the run.
runs = {"LLM no format line": [], "LLM schema": [], "Jev": []}
calls = {"LLM no format line": lambda text: ask_llm(text),
         "LLM schema": lambda text: ask_llm(text, schema=True),
         "Jev": ask_jev}
failed = []
for i, (text, label) in enumerate(sample, 1):
    for name, call in calls.items():
        try:
            runs[name].append({**call(text), "label": label, "message": text})
        except Exception as error:   # a timeout, or a 429 that outlived the retries
            failed.append((name, i, type(error).__name__))
    if i % 10 == 0:
        print(f"  {i}/{len(sample)} messages done, {len(failed)} failed calls")

print(f"\n{'':18} {'exact label':>11} {'accuracy':>9} {'p50 ms':>7} {'p95 ms':>7} {'$ per 1k':>9}  probabilities")
for name, rows in runs.items():
    exact = sum(r["team"] is not None for r in rows)
    accuracy = sum(r["team"] == r["label"] for r in rows) / len(rows)
    latencies = [r["ms"] for r in rows]
    p50 = statistics.median(latencies)
    p95 = statistics.quantiles(latencies, n=20)[18]
    per_1k = 1000 * statistics.mean(r["cost"] for r in rows)
    print(f"{name:18} {exact:>5}/{len(rows):<5} {accuracy:>9.1%} {p50:>7.0f} {p95:>7.0f} {per_1k:>9.3f}  "
          f"{'yes' if name == 'Jev' else 'no'}")
for name, i, kind in failed:
    print(f"  failed: {name} on message {i} ({kind})")
print("\nWith N = 40, one message is 2.5 points of accuracy: gaps of a few points are noise.")

In [ ]:
# ===========================================================================
# DEMO: what the reply without a format line looked like, and where Jev missed
# ===========================================================================
# Why: read the failures, not only the totals.
print("=== LLM, no format line: replies that are not a bare team name ===")
for r in runs["LLM no format line"]:
    if r["team"] is None:
        print(f"  {r['text'][:70]!r}")

print("\n=== JEV misses, with its confidence ===")
for r in runs["Jev"]:
    if r["team"] != r["label"]:
        print(f"  {r['message'][:45]:45} label {r['label']:18} jev {r['team']:18} conf {r['confidence']:.2f}")

> **Think About It**: Read your table, not mine. Is the schema row's accuracy within a few
> points of Jev's, or clearly apart? Count the Jev misses: how many are labels a person could
> argue about (is cancelling a restaurant reservation kitchen or travel?) and how many are
> plain mistakes? Then look at the last column: what would your code do with a probability
> per team that the LLM never gives you?
>
> **The honest part:** both rows use zero labelled examples, on 40 messages, in one run. We
> did not compare with a classifier trained on CLINC150's 15,000 labelled training messages,
> and with that much labelled data it is a strong baseline. Module 3 measures Jev against a
> baseline and on far more messages.

## Back to the slides: Accuracy within noise, 3 to 4x faster, 12x cheaper

## Demo 2: One request, three question types

One state, three questions, one call. Each question type comes back as its own typed answer,
and each question is answered on its own against the same state.

<!-- DIAGRAM: jev-request-anatomy -->
![One state and three questions (team Choice, frustration Score, travel_plans Noul) go to Jev in one request and come back as three typed answers](https://raw.githubusercontent.com/axelsirota/getting_started_jev/main/course/diagrams/module_01/jev-request-anatomy.png)

In [ ]:
# ===========================================================================
# DEMO: one state, three question types, one call (NARRATE)
# ===========================================================================
# Why: questions that share a state go in one request; the state is read, and paid for,
# once. (Level 3 of the lab measures what that buys you.)
# Criteria formats: a Choice takes a dict whose keys are the option names Jev sees and
# returns, and whose values describe each option (TEAMS above). A Score takes a list whose
# order is the scale: the first item is level 0, the last is the top level.
frustration_question = Score(
    instructions="How frustrated does the customer sound in `message`?",
    criteria=["the customer sounds calm",
              "the customer sounds annoyed",
              "the customer sounds angry or desperate"])
travel_question = Noul(instructions="Does `message` mention travel plans?")

start = time.perf_counter()
response = jev.system_one(state={"message": query}, model=JEV_MODEL, questions={
    "team": team_question,
    "frustration": frustration_question,
    "travel_plans": travel_question,
})
print(f"One call, three answers: {(time.perf_counter() - start) * 1000:.0f} ms, "
      f"{response.usage.input_tokens} input tokens, answered by {response.model}")
for name, answer in response.answers.items():
    print(f"  {name:13} {answer.type}")

In [ ]:
# ===========================================================================
# DEMO: read every field: choice, score, noul, probabilities, confidence (NARRATE)
# ===========================================================================
# Why: probabilities say how likely each option is. Confidence says how concentrated they
# are, with one formula per type (TypeSafe docs, Confidence):
#   Choice: (top probability - 1/n) / (1 - 1/n). 1.0: all on one option; 0.0: an even spread.
#   Score:  1 minus how far the probability sits from the most likely level, compared with
#           an even spread, floored at 0. Probability two levels away costs more than
#           probability on the next level, so a Score can show 0.00 with a clear top level.
#   Noul:   no confidence field; the noul is itself the probability.
# The score is the average level weighted by its probabilities. Confidence is not the same
# as being right; Module 3 tests that.
team = response.answers["team"]
frustration = response.answers["frustration"]
travel = response.answers["travel_plans"]

top3 = sorted(team.probabilities.items(), key=lambda kv: -kv[1])[:3]
print(f"team.choice       = {team.choice}   confidence {team.confidence:.2f}")
print("   top 3:", ", ".join(f"{name} {p:.2f}" for name, p in top3))
print(f"frustration.score = {frustration.score:.2f} on 0..2   confidence {frustration.confidence:.2f}")
for level, p in frustration.probabilities.items():
    print(f"   level {level}: {p:.2f}  {frustration.legend[level]}")
print(f"travel_plans.noul = {travel.noul:.2f}   (one number: the probability the statement is true)")

In [ ]:
# ===========================================================================
# DEMO: pinned versions: which model answered you? (NARRATE)
# ===========================================================================
# Why: "jev-latest" moves when TypeSafe ships a release; a pinned id does not. Every
# response says which version answered, so log it next to the decision.
# Today "jev-latest" may resolve to the same version as our pin, so both lines can match.
# Look at the model id on each line: that is the column that changes on release day.
for model_id in ("jev-latest", JEV_MODEL):
    r = jev.system_one(state={"message": query}, model=model_id,
                       questions={"travel_plans": travel_question})
    print(f"asked for {model_id:11} -> answered by {r.model:11}  travel_plans = {r.answers['travel_plans'].noul:.2f}")

> **Think About It**: Read your own frustration line above. What score did you get on the
> 0 to 2 scale, how much probability sits on each level, and what confidence? If confidence
> is near 0, the probability is spread across the levels: would you act on the score alone?
> And when `jev-latest` moves to the next release, which numbers in your code would you
> re-check before you follow it? (You are about to write some in the lab.)

## Back to the slides: Read the distribution, not only the top answer

## Lab 1: Triage in one call

<!-- DIAGRAM: triage-rules -->
![Jev judges urgency from 0 to 2 and risk from 0 to 1; triage() in your code checks the rules in order: risk at least 0.7 ESCALATE, risk above 0.3 REVIEW, urgency at least 1.5 PRIORITY, otherwise ROUTE](https://raw.githubusercontent.com/axelsirota/getting_started_jev/main/course/diagrams/module_01/triage-rules.png)

### Your Task

Build the first version of our triage layer: one Jev call per message returns the team, the
urgency and a risk signal, and your code turns those judgments into an action.

1. Write the urgency levels and the risk question (Level 1)
2. Write `triage()`: one Jev call, then the rules in the diagram (Level 2)
3. Optional: find out what bundling questions into one call really buys you (Level 3)

### Level 1: Guided

Fill `urgency_levels` (three situations, least to most urgent) and `risk_instructions` (one
statement Jev can judge true or false: the customer's money or account access is at risk).
Keep urgency about time and risk about money or access: they are different judgments. The
cell runs both questions on three messages. Stuck? The FALLBACK cell below fills in only the
slot you left empty, so you can move on.

### Level 2: Core

Write the body of `triage(message)`: one `jev.system_one` call with `team_question`,
`urgency_question` and `risk_question`, then the rules in the diagram, in order. Return the
team and its confidence, the urgency, the risk, the action and the model id that answered
(log it next to the decision). The cell runs it on the 12 lab messages, prints the risk
values low to high, then runs it on 10 CLINC150 messages so you can compare its team with
the dataset label.

Where the cut-offs come from: 0.7 and 0.3 are TypeSafe's suggested starting thresholds for a
Noul (above 0.7 reads as yes, below 0.3 as no, the band between is unsettled); 1.5 is our
choice for "needs doing within the hour". None of them is tuned on these 12 messages, and 12
messages could not tune them. Read the sorted risk values: where would you put the cut-offs,
and what would you measure to decide? Module 3 sets a review threshold from measured data.

### Level 3: Challenge (optional)

Write `compare_calls(messages, rounds)`: answer the three questions for each message in three
ways: one bundled call, three calls one after another, and three calls in parallel threads.
Rotate which way goes first for each message and repeat the pass `rounds` times, so no way
always pays for going first. Print p50 and p95 latency, mean input tokens and cost per 1,000
messages for each way. Then answer in a comment: what does bundling buy you, and when would
you split the calls anyway?

### Expected Output

- Level 1: the lost card scores high on urgency and above 0.7 on risk; the weather question
  scores near 0 on both
- Level 2: a 12-row table. In our run on 2026-10-30, with the solution's wording: 3 ESCALATE
  (lost card, unknown charge, someone used my card), 2 REVIEW (lost wallet, app login),
  1 PRIORITY (declined card), 6 ROUTE. Your own Level 1 wording can move a row; explain any
  row that differs. Then the sorted risk values and a 10-row label vs team table
- Level 3: three rows of numbers and your answer in a comment

### Hints

- Level 1: describe each Score level as a situation a person would recognize, not "more
  urgent than the previous level"
- Level 2: `response.answers["risk"].noul` is a float from 0 to 1;
  `response.answers["urgency"].score` is a float from 0 to 2; `response.model` names the
  model that answered
- Level 3: `time.perf_counter()`, `response.usage.input_tokens`, `statistics.median`,
  `statistics.quantiles(values, n=20)[18]` for p95, and `concurrent.futures.ThreadPoolExecutor`

In [ ]:
# ===========================================================================
# SOLUTION: LAB 1.1 (Guided)
# ===========================================================================
# Approach:
# - Each Score level is a situation in time a person recognizes on its own, ordered least to
#   most urgent, so the score reads 0 (can wait) to 2 (within the hour).
# - Risk is a separate judgment about money or access, phrased so that "true" means risk.
# Step 1: describe three urgency levels, least to most urgent, each as a situation in time.
urgency_levels = [
    "the customer is asking for information and nothing needs doing soon",
    "the customer needs something done in the next few days",
    "the customer needs something done within the hour",
]

# Step 2: one statement Jev can judge true or false: money or account access is at risk.
risk_instructions = ("Does `message` report a lost or stolen card, a charge the customer did "
                     "not make, or a locked account?")

urgency_question = Score(instructions="How urgent is `message` for the customer?",
                         criteria=urgency_levels)
risk_question = Noul(instructions=risk_instructions)
for text in ["i lost my card at the mall and need it frozen right away",
             "there's a charge on my statement i never made",
             "will it rain tomorrow"]:
    r = jev.system_one(state={"message": text}, model=JEV_MODEL,
                       questions={"urgency": urgency_question, "risk": risk_question})
    print(f"{text[:50]:50} urgency {r.answers['urgency'].score:.2f}   risk {r.answers['risk'].noul:.2f}")
# Explanation:
# - Levels written as "more urgent than the last one" give Jev nothing to recognize.
# - If urgency level 2 said "money is at risk", urgency and risk would be one judgment asked
#   twice, and the declined card (urgent, not risky) could no longer be told apart.
# - Our risk statement joins three conditions with "or". It works, and Module 2 shows why
#   three separate Nouls are better: you learn WHICH risk fired.

In [ ]:
# ===========================================================================
# SOLUTION: LAB 1.2 (Core)
# ===========================================================================
# Approach:
# - One call carries all three questions: the state is read once.
# - Jev returns judgments; the rules that turn them into an action are plain Python, in the
#   order of the diagram, so you can read, test and change every threshold.
# - The result also carries the team confidence and the model id, so a log line can say how
#   sure Jev was and which version answered.
LAB_MESSAGES = [
    "i lost my card at the mall and need it frozen right away",
    "there's a charge on my statement i never made",
    "i think someone used my card but i'm not sure",
    "i can't find my wallet, maybe it's in the car",
    "the bank app won't let me log in",
    "why was my card declined at the gas station",
    "what's my checking balance",
    "how do i make lasagna",
    "will it rain tomorrow",
    "tell me a joke",
    "i need to book a flight to boston next month",
    "what movies are playing",
]


def triage(message: str) -> dict:
    """Triage one message with a single Jev call.

    Ask team_question, urgency_question and risk_question in one jev.system_one call, then
    apply the rules in the diagram above, in order, to pick the action.

    Returns:
        {"team", "team_confidence", "urgency", "risk", "action", "model"}
    """
    response = jev.system_one(state={"message": message}, model=JEV_MODEL, questions={
        "team": team_question,
        "urgency": urgency_question,
        "risk": risk_question,
    })
    team = response.answers["team"]
    urgency = response.answers["urgency"].score
    risk = response.answers["risk"].noul
    if risk >= 0.7:
        action = "ESCALATE"
    elif risk > 0.3:
        action = "REVIEW"
    elif urgency >= 1.5:
        action = "PRIORITY"
    else:
        action = "ROUTE"
    return {"team": team.choice, "team_confidence": team.confidence, "urgency": urgency,
            "risk": risk, "action": action, "model": response.model}


print(f"{'message':44} {'team':18} {'conf':>4} {'urg':>5} {'risk':>5}  action")
results = []
for text in LAB_MESSAGES:
    result = triage(text)
    if result is None:
        print("triage() returned None: write its body above.")
        break
    results.append(result)
    print(f"{text[:44]:44} {result['team']:18} {result['team_confidence']:>4.2f} "
          f"{result['urgency']:>5.2f} {result['risk']:>5.2f}  {result['action']}")

if results:
    print(f"\nAnswered by {results[-1]['model']}. Risk, low to high: where would you cut?")
    print("  " + "  ".join(f"{r['risk']:.2f}" for r in sorted(results, key=lambda r: r["risk"])))
    print("\n=== 10 CLINC150 messages, one per team: label vs triage() ===")
    for text, label in sample[::4]:
        r = triage(text)
        mark = "OK  " if r["team"] == label else "MISS"
        print(f"  [{mark}] {text[:40]:40} label {label:18} team {r['team']:18} {r['action']}")
# Explanation:
# - In our run on 2026-10-30 the declined card was urgent (about 1.8) but not a risk (about
#   0.04): PRIORITY, not ESCALATE. That row is why urgency and risk are separate questions.
# - The lost wallet (risk about 0.36) and the app login (about 0.50) landed in the band where
#   the question does not settle it: a person reviews them.
# - "what movies are playing" fits no team, yet the Choice still picks one: read its team
#   confidence in your table (0.49 and 0.42 in two runs). Module 3 adds the question "does any team fit?".
# - 0.7 and 0.3 are TypeSafe's starting thresholds for a Noul, not fitted to these rows; the
#   sorted risk line shows how few messages sit near them. Module 3 measures them on data.
# - Common mistake: asking Jev "should this be escalated?". That hides the rule inside the
#   model; when the policy changes you cannot change a threshold, you rewrite a prompt.

In [ ]:
# ===========================================================================
# SOLUTION: LAB 1.3 (Challenge, optional)
# ===========================================================================
# Approach:
# - Time the three ways on the same messages; read usage.input_tokens from every response.
# - Rotate which way goes first for each message and repeat the pass, so no way always runs
#   first (or always second) and order effects average out.
# Spec: compare_calls(messages, rounds) answers team, urgency and risk for every message
# three ways:
#   A) one call with all three questions
#   B) three calls, one question each, one after another
#   C) three calls, one question each, in parallel threads
# Rotate which way goes first for each message, and repeat the whole pass `rounds` times.
# Print, for A, B and C: p50 and p95 latency in ms, mean input tokens per message, and cost
# per 1,000 messages (JEV_PRICE_IN).
# Then, in a comment below the call: what does bundling buy you, and when would you split?
from concurrent.futures import ThreadPoolExecutor


def compare_calls(messages: list[str], rounds: int = 2) -> None:
    questions = {"team": team_question, "urgency": urgency_question, "risk": risk_question}

    def one(message, name, question):
        return jev.system_one(state={"message": message}, model=JEV_MODEL,
                              questions={name: question})

    def bundled(message):
        return [jev.system_one(state={"message": message}, model=JEV_MODEL, questions=questions)]

    def sequential(message):
        return [one(message, n, q) for n, q in questions.items()]

    def parallel(message):
        return list(pool.map(lambda nq: one(message, *nq), questions.items()))

    ways = [("A bundled", bundled), ("B sequential", sequential), ("C parallel", parallel)]
    ms = {way: [] for way, _ in ways}
    tokens = {way: [] for way, _ in ways}
    with ThreadPoolExecutor(max_workers=3) as pool:
        for round_number in range(rounds):
            for i, message in enumerate(messages):
                shift = (i + round_number) % len(ways)   # who goes first rotates
                for way, call in ways[shift:] + ways[:shift]:
                    start = time.perf_counter()
                    responses = call(message)
                    ms[way].append((time.perf_counter() - start) * 1000)
                    tokens[way].append(sum(r.usage.input_tokens for r in responses))
    for way in ms:
        p50 = statistics.median(ms[way])
        p95 = statistics.quantiles(ms[way], n=20)[18]
        mean_tokens = statistics.mean(tokens[way])
        per_1k = 1000 * mean_tokens * JEV_PRICE_IN
        print(f"{way:13} p50 {p50:5.0f} ms   p95 {p95:5.0f} ms   "
              f"input tokens {mean_tokens:5.0f}   $ per 1k {per_1k:.3f}")


compare_calls(LAB_MESSAGES[:8])
# Explanation:
# - Measured 2026-10-30, 8 messages, 2 rounds, order rotated: p50 A 247 ms, B 735 ms,
#   C 262 ms; p95 302, 809 and 510 ms; 632 vs 1,180 input tokens ($0.027 vs $0.050 per 1,000).
# - In that run parallel calls were about as fast as one bundled call at p50 and slower at
#   p95, so bundling bought cost and a steadier tail, not a faster median: the state is read, and paid for, once instead of three times. That is
#   what we measured from one laptop, not a claim about how Jev schedules questions inside.
# - Split the calls when code needs one answer to build the next state, or when a field in
#   the state that one question needs misleads another.

## Back to the slides: Urgent is not risky: the declined card

# Summary

## Key Takeaways

1. **Jev answers typed questions about a state.** Choice, Score and Noul come back as data
   your code reads, not text it parses.
2. **Same accuracy, different contract.** In our run on 2026-10-30 (40 messages, one run),
   the LLM with a JSON schema and Jev were within noise on accuracy; Jev was about 4x faster
   (252 vs 946 ms p50), about 12x cheaper ($0.022 vs $0.256 per 1,000 messages), and
   returned a probability per team. Check your own table: your numbers will move a little.
3. **Jev judges, your code decides.** The escalation rule lives in `triage()`, where you can
   read it, test it and change a threshold.

## What This Module Leaves Open

| Gap | Where |
|---|---|
| Which risk fired: our risk question joins three conditions with "or" | Module 2 |
| Messages that fit no team: a Choice always picks one | Module 3 |
| Thresholds checked on held-out data, and a baseline to beat | Module 3 |
| Rate limits, timeouts and failed calls under real traffic | Not in this course: start from the SDK retry policy and the TypeSafe docs, Models |

## Looking Ahead

Module 2, Designing Good Questions: our risk question joins three conditions with "or", so
when it fires we cannot tell which risk it was, and the team question stumbles on restaurant
reservations. We split fuzzy judgments into atomic questions and move logic and dates into code.

## Homework

Write 10 messages like the ones your own support inbox gets (no real customer data) and run
`triage()` on them. Which rule fires most? Pick one answer you disagree with: is the fix in
the state, the question, or the code?